<a href="https://colab.research.google.com/github/Sayana-mc/agentic-resume-screening-ai/blob/main/Resume_Screening_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
chroma_client.delete_collection(name="resumes")
collection = chroma_client.get_or_create_collection(name="resumes")

In [1]:
!pip install pymupdf python-docx google-generativeai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7

In [61]:
from google.colab import userdata
import google.generativeai as genai

api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)

llm_model = genai.GenerativeModel('models/gemini-3.5-flash')
print("Gemini loaded!")

Gemini loaded!


In [62]:
for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [63]:
from google.colab import files
uploaded = files.upload()

Saving resume1.docx to resume1 (5).docx
Saving resume2.docx to resume2 (4).docx
Saving resume3.docx to resume3 (5).docx


In [64]:
import fitz
from docx import Document

def extract_text(file_path):
    if file_path.endswith(".pdf"):
        doc = fitz.open(file_path)
        return "".join([page.get_text() for page in doc])
    elif file_path.endswith(".docx"):
        doc = Document(file_path)
        return "\n".join([p.text for p in doc.paragraphs])
    return ""

resume_files = list(uploaded.keys())
print("Uploaded:", resume_files)

Uploaded: ['resume1 (5).docx', 'resume2 (4).docx', 'resume3 (5).docx']


In [65]:
def chunk_text(text, chunk_size=300):
    """Text-നെ ~300 words ഉള്ള chunks ആയി split ചെയ്യുന്നു"""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

# to test
sample_text = extract_text(resume_files[0])
sample_chunks = chunk_text(sample_text)
print(f"Total chunks: {len(sample_chunks)}")
print("First chunk:\n", sample_chunks[0])

Total chunks: 1
First chunk:
 Aisha Kapoor aisha.kapoor.dev@email.com | +91-98765-43210 | Bangalore, India LinkedIn: linkedin.com/in/aishakapoordev SUMMARY Backend software engineer with 5 years of experience building scalable web applications and REST APIs. Strong background in Python, cloud infrastructure, and distributed systems. AWS Certified Solutions Architect. SKILLS Python, Django, FastAPI, PostgreSQL, Redis, Docker, Kubernetes, AWS (EC2, S3, Lambda, RDS), CI/CD (Jenkins, GitHub Actions), Microservices Architecture, System Design, Git WORK EXPERIENCE Senior Backend Engineer — Nimbus Cloud Solutions June 2022 – Present - Led migration of a monolithic application to microservices, reducing average API response time by 40% - Designed and implemented a REST API serving 2M+ daily requests using FastAPI and PostgreSQL - Mentored 2 junior engineers and conducted code reviews - Set up CI/CD pipelines using GitHub Actions, cutting deployment time from 45 minutes to 8 minutes Backend Dev

In [66]:
import chromadb
import google.generativeai as genai

# ChromaDB client  (in-memory, temporary)
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="resumes")

def get_embedding(text):
    result = genai.embed_content(
        model="models/gemini-embedding-001",
        content=text,
        task_type="retrieval_document"
    )
    return result['embedding']

# all resumes chunk  embeddings  ChromaDB store
chunk_id = 0
for filename in resume_files:
    text = extract_text(filename)
    chunks = chunk_text(text)
    for chunk in chunks:
        embedding = get_embedding(chunk)
        collection.add(
            ids=[f"chunk_{chunk_id}"],
            embeddings=[embedding],
            documents=[chunk],
            metadatas=[{"filename": filename}]
        )
        chunk_id += 1

print(f"Total chunks stored in vector DB: {chunk_id}")

Total chunks stored in vector DB: 3


In [67]:
job_description = """
We are hiring a Backend Software Engineer with 3+ years of experience in
Python, Django or FastAPI, and cloud platforms like AWS. Experience with
REST APIs, PostgreSQL, and Docker is required.
"""
print("Job description set!")

Job description set!


In [68]:
def retrieve_relevant_chunks(query_text, filename, top_k=3):
    query_embedding = genai.embed_content(
        model="models/gemini-embedding-001",
        content=query_text,
        task_type="retrieval_query"
    )['embedding']

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        where={"filename": filename}
    )
    return " ".join(results['documents'][0])

# testing
relevant = retrieve_relevant_chunks(job_description, resume_files[0])
print("Most relevant resume content for this JD:\n", relevant[:500])

Most relevant resume content for this JD:
 Aisha Kapoor aisha.kapoor.dev@email.com | +91-98765-43210 | Bangalore, India LinkedIn: linkedin.com/in/aishakapoordev SUMMARY Backend software engineer with 5 years of experience building scalable web applications and REST APIs. Strong background in Python, cloud infrastructure, and distributed systems. AWS Certified Solutions Architect. SKILLS Python, Django, FastAPI, PostgreSQL, Redis, Docker, Kubernetes, AWS (EC2, S3, Lambda, RDS), CI/CD (Jenkins, GitHub Actions), Microservices Architecture, 


In [69]:
import json

def evaluation_agent(relevant_content, jd_text):
    prompt = f"""
You are an experienced recruiter. Based on the retrieved resume evidence below,
evaluate the candidate's fit for this job.

JOB DESCRIPTION:
{jd_text}

RETRIEVED RESUME EVIDENCE:
{relevant_content}

Respond ONLY with valid JSON, no markdown, no extra text:
{{
  "score": <0-100>,
  "strengths": ["...", "..."],
  "gaps": ["...", "..."],
  "verdict": "Shortlist" or "Borderline" or "Reject"
}}
"""
    response = llm_model.generate_content(prompt)
    text = response.text.strip()
    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

print("Evaluation agent ready!")

Evaluation agent ready!


In [70]:
def interview_question_agent(strengths, gaps, jd_text):
    prompt = f"""
You are an interview panel expert. Based on this candidate's strengths and gaps
for the given job, generate 3 targeted interview questions.

JOB DESCRIPTION: {jd_text}
STRENGTHS: {strengths}
GAPS: {gaps}

Respond ONLY with valid JSON, no markdown:
{{
  "questions": ["question1", "question2", "question3"]
}}
"""
    response = llm_model.generate_content(prompt)
    text = response.text.strip()
    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

print("Interview agent ready!")

Interview agent ready!


In [71]:
final_results = []

for filename in resume_files:
    print(f"\n🔄 Processing {filename}...")

    # Step 1: RAG Retrieval
    relevant_content = retrieve_relevant_chunks(job_description, filename)

    # Step 2: Evaluation
    eval_result = evaluation_agent(relevant_content, job_description)
    eval_result["filename"] = filename

    # Step 3: Orchestrator Decision Logic
    if eval_result["score"] >= 70:
        eval_result["action"] = "Auto-Shortlisted"
        # Step 4: Interview Agent trigger
        interview_data = interview_question_agent(
            eval_result["strengths"], eval_result["gaps"], job_description
        )
        eval_result["interview_questions"] = interview_data["questions"]
    elif eval_result["score"] >= 50:
        eval_result["action"] = "Needs Human Review"
        eval_result["interview_questions"] = []
    else:
        eval_result["action"] = "Auto-Rejected"
        eval_result["interview_questions"] = []

    final_results.append(eval_result)

print("\n✅ All resumes processed!")


🔄 Processing resume1 (5).docx...

🔄 Processing resume2 (4).docx...

🔄 Processing resume3 (5).docx...

✅ All resumes processed!


In [72]:
final_results.sort(key=lambda x: x["score"], reverse=True)

for r in final_results:
    print("="*60)
    print(f"📄 {r['filename']}")
    print(f"   Score: {r['score']}/100 | Action: {r['action']}")
    print(f"   ✅ Strengths: {', '.join(r['strengths'])}")
    print(f"   ⚠️ Gaps: {', '.join(r['gaps'])}")
    if r["interview_questions"]:
        print(f"   ❓ Interview Questions:")
        for q in r["interview_questions"]:
            print(f"      - {q}")
    print()

📄 resume1 (5).docx
   Score: 98/100 | Action: Auto-Shortlisted
   ✅ Strengths: Over 4 years of professional backend development experience, exceeding the 3-year requirement., Strong expertise in both required frameworks: FastAPI and Django., AWS Certified Solutions Architect with practical experience using EC2, S3, Lambda, and RDS., Proven track record of designing high-throughput REST APIs (serving 2M+ daily requests)., Direct experience with PostgreSQL, Docker, and CI/CD pipelines.
   ⚠️ Gaps: 
   ❓ Interview Questions:
      - You have a proven track record of designing REST APIs serving over 2 million daily requests. Can you describe a specific performance bottleneck you encountered—either in your Python framework (FastAPI/Django) or the PostgreSQL database—and how you resolved it?
      - Given your background as an AWS Certified Solutions Architect, how would you leverage services like ECS, Lambda, and RDS to build a highly available and cost-efficient deployment pipeline for a D

In [73]:
import re

def extract_email(text):
    match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    return match.group(0) if match else None

# to test
test_text = extract_text(resume_files[0])
print("Found email:", extract_email(test_text))

Found email: aisha.kapoor.dev@email.com


In [76]:
import smtplib
from email.mime.text import MIMEText
from google.colab import userdata

sender_email = userdata.get('SENDER_EMAIL')
sender_password = userdata.get('SENDER_APP_PASSWORD')

def communication_agent(candidate_email, candidate_name, action, jd_role="the position"):
    if candidate_email is None:
        print("  ⚠️ No email found, skipping.")
        return

    if action == "Auto-Shortlisted":
        subject = f"Great news regarding your application for {jd_role}"
        body = f"""Dear {candidate_name},

Thank you for applying. We are pleased to inform you that your profile has
been shortlisted for the next round of the hiring process for {jd_role}.

Our team will reach out shortly to schedule an interview.

Best regards,
Hiring Team
"""
    elif action == "Needs Human Review":
        subject = f"Update on your application for {jd_role}"
        body = f"""Dear {candidate_name},

Thank you for applying for {jd_role}. Your application is currently under
review by our hiring team. We will get back to you soon with an update.

Best regards,
Hiring Team
"""
    else:  # Auto-Rejected
        subject = f"Update on your application for {jd_role}"
        body = f"""Dear {candidate_name},

Thank you for your interest in {jd_role} and for taking the time to apply.
After careful review, we have decided not to move forward with your
application at this time.

We wish you the best in your job search.

Best regards,
Hiring Team
"""

    msg = MIMEText(body)
    msg['Subject'] = subject
    msg['From'] = sender_email
    msg['To'] = candidate_email

    try:
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, candidate_email, msg.as_string())
        server.quit()
        print(f"  📧 Email sent to {candidate_email} ({action})")
    except Exception as e:
        print(f"  ❌ Failed to send email: {e}")

print("Communication agent ready!")

Communication agent ready!


In [77]:
import time

final_results = []

for filename in resume_files:
    print(f"\n🔄 Processing {filename}...")

    text = extract_text(filename)
    candidate_email = extract_email(text)

    # Name extract ചെയ്യാൻ simple ആയി — resume-ലെ ആദ്യ line ഉപയോഗിക്കാം
    candidate_name = text.strip().split("\n")[0][:50]

    # Step 1: RAG Retrieval
    relevant_content = retrieve_relevant_chunks(job_description, filename)

    # Step 2: Evaluation (with retry)
    eval_result = None
    for attempt in range(3):
        try:
            eval_result = evaluation_agent(relevant_content, job_description)
            break
        except Exception as e:
            print(f"  Retry {attempt+1} after error: {e}")
            time.sleep(20)

    if eval_result is None:
        print(f"  Skipping {filename} — failed after retries")
        continue

    eval_result["filename"] = filename
    eval_result["email"] = candidate_email

    # Step 3: Orchestrator Decision Logic
    if eval_result["score"] >= 70:
        eval_result["action"] = "Auto-Shortlisted"
        time.sleep(3)
        interview_data = interview_question_agent(
            eval_result["strengths"], eval_result["gaps"], job_description
        )
        eval_result["interview_questions"] = interview_data["questions"]
    elif eval_result["score"] >= 50:
        eval_result["action"] = "Needs Human Review"
        eval_result["interview_questions"] = []
    else:
        eval_result["action"] = "Auto-Rejected"
        eval_result["interview_questions"] = []

    # Step 4: Communication Agent — automatic email
    communication_agent(candidate_email, candidate_name, eval_result["action"], "Backend Software Engineer")

    final_results.append(eval_result)
    time.sleep(5)

print("\n✅ All resumes processed and emails sent!")


🔄 Processing resume1 (5).docx...
  📧 Email sent to aisha.kapoor.dev@email.com (Auto-Rejected)

🔄 Processing resume2 (4).docx...
  📧 Email sent to rohan.mehta.job@email.com (Auto-Rejected)

🔄 Processing resume3 (5).docx...
  📧 Email sent to sneha.iyer.contact@email.com (Auto-Rejected)

✅ All resumes processed and emails sent!
